In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import evaluate
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from transformers import BlipProcessor, BlipForConditionalGeneration

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f"ONNX Runtime: {ort.__version__}")


c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0530 22:39:09.880000 16072 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


ONNX Runtime: 1.26.0


In [ ]:
ROOT_DIR    = "../data_ready_for_kaggle"
TEST_PATH   = os.path.join(ROOT_DIR, "cleaned_test.csv")
IMAGES_DIR  = os.path.join(ROOT_DIR, "images_resized")
MODEL_DIR   = os.path.abspath("../saved_models_v3")
RESULTS_DIR = "./results"
SAMPLE_IMAGE = "../data_ready_for_kaggle/images_resized/00000006091.jpg"

FP32_DIR              = "./model_quantized/onnx_fp32"
ENCODER_FP32_PATH     = os.path.join(FP32_DIR, 'vision_encoder_fp32.onnx')
DEC_INIT_FP32_PATH    = os.path.join(FP32_DIR, 'text_decoder_init_fp32.onnx')
DEC_PAST_FP32_PATH    = os.path.join(FP32_DIR, 'text_decoder_with_past_fp32.onnx')

INT8_DIR              = "./model_quantized/onnx_int8"
os.makedirs(INT8_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

ENCODER_INT8_PATH     = ENCODER_FP32_PATH
DEC_INIT_INT8_PATH    = os.path.join(INT8_DIR, 'text_decoder_init_int8.onnx')
DEC_PAST_INT8_PATH    = os.path.join(INT8_DIR, 'text_decoder_with_past_int8.onnx')

MAX_TEST_IMAGES = 200
MAX_LENGTH      = 64
BATCH_SIZE      = 16
NUM_WORKERS     = 0
WARMUP_STEPS    = 3

for p in [ENCODER_FP32_PATH, DEC_INIT_FP32_PATH, DEC_PAST_FP32_PATH]:
    assert os.path.exists(p), f"File không tìm thấy: {p}\n→ Chạy onnx_export_fp32.ipynb trước!"
    size_mb = os.path.getsize(p) / 1e6
    print(f"OK  {os.path.basename(p):45s}  {size_mb:7.1f} MB")


OK  vision_encoder_fp32.onnx                         344.5 MB
OK  text_decoder_init_fp32.onnx                      851.5 MB
OK  text_decoder_with_past_fp32.onnx                 794.8 MB


In [ ]:
processor = BlipProcessor.from_pretrained(MODEL_DIR)
model     = BlipForConditionalGeneration.from_pretrained(MODEL_DIR)
model.eval()

bos_token_id = model.config.text_config.bos_token_id
eos_token_id = model.config.text_config.eos_token_id
pad_token_id = model.config.text_config.pad_token_id
if pad_token_id is None:
    pad_token_id = eos_token_id

num_layers      = model.text_decoder.config.num_hidden_layers
patch_size      = model.vision_model.config.patch_size
image_size      = model.vision_model.config.image_size
encoder_seq_len = (image_size // patch_size) ** 2 + 1

print(f"num_layers     : {num_layers}")
print(f"encoder_seq_len: {encoder_seq_len}")


Loading weights: 100%|██████████| 471/471 [00:00<00:00, 2015.10it/s]

num_layers     : 12
encoder_seq_len: 577


In [ ]:
Q
U
A
N
T
_
C
O
N
F
I
G

=

d
i
c
t
(





w
e
i
g
h
t
_
t
y
p
e


=

Q
u
a
n
t
T
y
p
e
.
Q
I
n
t
8
,





r
e
d
u
c
e
_
r
a
n
g
e

=

T
r
u
e
,













s
a
f
e

c
h
o

I
n
t
e
l

C
P
U

k
h
ô
n
g

c
ó

V
N
N
I





e
x
t
r
a
_
o
p
t
i
o
n
s
=

{
'
M
a
t
M
u
l
C
o
n
s
t
B
O
n
l
y
'
:

T
r
u
e
}
,



w
e
i
g
h
t
-
o
n
l
y
,

a
c
t
i
v
a
t
i
o
n

F
P
3
2






N
O
T
E
:

`
o
p
t
i
m
i
z
e
_
m
o
d
e
l
`

k
h
ô
n
g

p
h
ả
i

t
h
a
m

s
ố

c
ủ
a

q
u
a
n
t
i
z
e
_
d
y
n
a
m
i
c

(
o
r
t

1
.
2
6
+
)






G
r
a
p
h

o
p
t
i
m
i
z
a
t
i
o
n

đ
ư
ợ
c

b
ậ
t

q
u
a

S
e
s
s
i
o
n
O
p
t
i
o
n
s

k
h
i

l
o
a
d

s
e
s
s
i
o
n
.

)


p
r
i
n
t
(
"
Q
u
a
n
t
i
z
i
n
g

t
e
x
t
_
d
e
c
o
d
e
r
_
i
n
i
t
.
.
.
"
)

q
u
a
n
t
i
z
e
_
d
y
n
a
m
i
c
(
D
E
C
_
I
N
I
T
_
F
P
3
2
_
P
A
T
H
,

D
E
C
_
I
N
I
T
_
I
N
T
8
_
P
A
T
H
,

*
*
Q
U
A
N
T
_
C
O
N
F
I
G
)

p
r
i
n
t
(
f
"


S
a
v
e
d
:

{
D
E
C
_
I
N
I
T
_
I
N
T
8
_
P
A
T
H
}
"
)


p
r
i
n
t
(
"
Q
u
a
n
t
i
z
i
n
g

t
e
x
t
_
d
e
c
o
d
e
r
_
w
i
t
h
_
p
a
s
t
.
.
.
"
)

q
u
a
n
t
i
z
e
_
d
y
n
a
m
i
c
(
D
E
C
_
P
A
S
T
_
F
P
3
2
_
P
A
T
H
,

D
E
C
_
P
A
S
T
_
I
N
T
8
_
P
A
T
H
,

*
*
Q
U
A
N
T
_
C
O
N
F
I
G
)

p
r
i
n
t
(
f
"


S
a
v
e
d
:

{
D
E
C
_
P
A
S
T
_
I
N
T
8
_
P
A
T
H
}
"
)


p
r
i
n
t
(
"
\
n
[
V
i
s
i
o
n

e
n
c
o
d
e
r

g
i
ữ

F
P
3
2

—

k
h
ô
n
g

q
u
a
n
t
i
z
e
]
"
)



R
e
p
o
r
t

s
i
z
e

r
e
d
u
c
t
i
o
n

p
r
i
n
t
(
"
\
n
-
-
-

S
i
z
e

c
o
m
p
a
r
i
s
o
n

(
T
e
x
t

D
e
c
o
d
e
r
)

-
-
-
"
)

f
o
r

f
p
,

i
p
,

l
a
b
e
l

i
n

[





(
D
E
C
_
I
N
I
T
_
F
P
3
2
_
P
A
T
H
,


D
E
C
_
I
N
I
T
_
I
N
T
8
_
P
A
T
H
,


'
d
e
c
o
d
e
r
_
i
n
i
t
'
)
,





(
D
E
C
_
P
A
S
T
_
F
P
3
2
_
P
A
T
H
,


D
E
C
_
P
A
S
T
_
I
N
T
8
_
P
A
T
H
,


'
d
e
c
o
d
e
r
_
w
i
t
h
_
p
a
s
t
'
)
,

]
:





f
p
_
m
b

=

o
s
.
p
a
t
h
.
g
e
t
s
i
z
e
(
f
p
)

/

1
e
6





i
p
_
m
b

=

o
s
.
p
a
t
h
.
g
e
t
s
i
z
e
(
i
p
)

/

1
e
6





r
a
t
i
o

=

f
p
_
m
b

/

i
p
_
m
b





p
r
i
n
t
(
f
"


{
l
a
b
e
l
:
2
5
s
}


F
P
3
2
=
{
f
p
_
m
b
:
.
1
f
}
M
B


I
N
T
8
=
{
i
p
_
m
b
:
.
1
f
}
M
B


r
a
t
i
o
=
{
r
a
t
i
o
:
.
2
f
}
x
"
)


In [ ]:
c
l
a
s
s

U
I
T
V
i
I
C
D
a
t
a
s
e
t
(
D
a
t
a
s
e
t
)
:





d
e
f

_
_
i
n
i
t
_
_
(
s
e
l
f
,

d
a
t
a
_
f
i
l
e
,

i
m
g
_
d
i
r
,

p
r
o
c
e
s
s
o
r
,

m
a
x
_
l
e
n
g
t
h
=
6
4
)
:









d
f

=

p
d
.
r
e
a
d
_
c
s
v
(
d
a
t
a
_
f
i
l
e
)









s
e
l
f
.
i
m
g
_
d
i
r




=

i
m
g
_
d
i
r









s
e
l
f
.
p
r
o
c
e
s
s
o
r


=

p
r
o
c
e
s
s
o
r









s
e
l
f
.
m
a
x
_
l
e
n
g
t
h

=

m
a
x
_
l
e
n
g
t
h









s
e
l
f
.
i
m
a
g
e
_
g
r
o
u
p
s

=

(













d
f
.
g
r
o
u
p
b
y
(
'
i
m
a
g
e
_
f
i
l
e
'
)
[
'
c
a
p
t
i
o
n
'
]
.
a
p
p
l
y
(
l
i
s
t
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)









)






d
e
f

_
_
l
e
n
_
_
(
s
e
l
f
)
:









r
e
t
u
r
n

l
e
n
(
s
e
l
f
.
i
m
a
g
e
_
g
r
o
u
p
s
)






d
e
f

_
_
g
e
t
i
t
e
m
_
_
(
s
e
l
f
,

i
d
x
)
:









r
o
w








=

s
e
l
f
.
i
m
a
g
e
_
g
r
o
u
p
s
.
i
l
o
c
[
i
d
x
]









i
m
a
g
e
_
f
i
l
e

=

r
o
w
[
'
i
m
a
g
e
_
f
i
l
e
'
]









i
m
a
g
e

=

I
m
a
g
e
.
o
p
e
n
(
o
s
.
p
a
t
h
.
j
o
i
n
(
s
e
l
f
.
i
m
g
_
d
i
r
,

i
m
a
g
e
_
f
i
l
e
)
)
.
c
o
n
v
e
r
t
(
'
R
G
B
'
)









e
n
c

=

s
e
l
f
.
p
r
o
c
e
s
s
o
r
(













i
m
a
g
e
s
=
i
m
a
g
e
,

r
e
t
u
r
n
_
t
e
n
s
o
r
s
=
'
p
t
'
,













p
a
d
d
i
n
g
=
'
m
a
x
_
l
e
n
g
t
h
'
,

t
r
u
n
c
a
t
i
o
n
=
T
r
u
e
,

m
a
x
_
l
e
n
g
t
h
=
s
e
l
f
.
m
a
x
_
l
e
n
g
t
h
,









)









e
n
c

=

{
k
:

v
.
s
q
u
e
e
z
e
(
0
)

f
o
r

k
,

v

i
n

e
n
c
.
i
t
e
m
s
(
)
}









e
n
c
[
'
i
m
a
g
e
_
f
i
l
e
'
]



=

i
m
a
g
e
_
f
i
l
e









e
n
c
[
'
r
a
w
_
c
a
p
t
i
o
n
s
'
]

=

r
o
w
[
'
c
a
p
t
i
o
n
'
]









r
e
t
u
r
n

e
n
c



d
e
f

c
o
l
l
a
t
e
_
f
n
(
b
a
t
c
h
)
:





o
u
t

=

{
k
:

t
o
r
c
h
.
s
t
a
c
k
(
[
b
[
k
]

f
o
r

b

i
n

b
a
t
c
h
]
)

f
o
r

k

i
n

[
'
p
i
x
e
l
_
v
a
l
u
e
s
'
]
}





o
u
t
[
'
i
m
a
g
e
_
f
i
l
e
'
]



=

[
b
[
'
i
m
a
g
e
_
f
i
l
e
'
]



f
o
r

b

i
n

b
a
t
c
h
]





o
u
t
[
'
r
a
w
_
c
a
p
t
i
o
n
s
'
]

=

[
b
[
'
r
a
w
_
c
a
p
t
i
o
n
s
'
]

f
o
r

b

i
n

b
a
t
c
h
]





r
e
t
u
r
n

o
u
t



d
e
f

b
u
i
l
d
_
r
e
f
e
r
e
n
c
e
s
(
d
a
t
a
l
o
a
d
e
r
)
:





r
e
f
s

=

{
}





f
o
r

b
a
t
c
h

i
n

d
a
t
a
l
o
a
d
e
r
:









f
o
r

i
m
g
,

c
a
p
s

i
n

z
i
p
(
b
a
t
c
h
[
'
i
m
a
g
e
_
f
i
l
e
'
]
,

b
a
t
c
h
[
'
r
a
w
_
c
a
p
t
i
o
n
s
'
]
)
:













r
e
f
s
[
i
m
g
]

=

c
a
p
s





r
e
t
u
r
n

r
e
f
s



d
e
f

c
a
l
c
u
l
a
t
e
_
m
e
t
r
i
c
s
(
p
r
e
d
s
_
d
i
c
t
,

r
e
f
s
_
d
i
c
t
)
:





c
o
m
m
o
n

=

s
o
r
t
e
d
(
s
e
t
(
p
r
e
d
s
_
d
i
c
t
)

&

s
e
t
(
r
e
f
s
_
d
i
c
t
)
)





p
r
e
d
s


=

[
p
r
e
d
s
_
d
i
c
t
[
k
]

f
o
r

k

i
n

c
o
m
m
o
n
]





r
e
f
s



=

[
r
e
f
s
_
d
i
c
t
[
k
]


f
o
r

k

i
n

c
o
m
m
o
n
]





b
l
e
u



=

e
v
a
l
u
a
t
e
.
l
o
a
d
(
'
b
l
e
u
'
)





r
o
u
g
e


=

e
v
a
l
u
a
t
e
.
l
o
a
d
(
'
r
o
u
g
e
'
)





m
e
t
e
o
r

=

e
v
a
l
u
a
t
e
.
l
o
a
d
(
'
m
e
t
e
o
r
'
)





r
e
t
u
r
n

{









'
b
l
e
u
4
'
:


b
l
e
u
.
c
o
m
p
u
t
e
(
p
r
e
d
i
c
t
i
o
n
s
=
p
r
e
d
s
,

r
e
f
e
r
e
n
c
e
s
=
r
e
f
s
)
[
'
b
l
e
u
'
]

*

1
0
0
,









'
r
o
u
g
e
L
'
:

r
o
u
g
e
.
c
o
m
p
u
t
e
(
p
r
e
d
i
c
t
i
o
n
s
=
p
r
e
d
s
,

r
e
f
e
r
e
n
c
e
s
=
r
e
f
s
)
[
'
r
o
u
g
e
L
'
]

*

1
0
0
,









'
m
e
t
e
o
r
'
:

m
e
t
e
o
r
.
c
o
m
p
u
t
e
(
p
r
e
d
i
c
t
i
o
n
s
=
p
r
e
d
s
,

r
e
f
e
r
e
n
c
e
s
=
r
e
f
s
)
[
'
m
e
t
e
o
r
'
]

*

1
0
0
,









'
n
u
m
_
i
m
a
g
e
s
'
:

l
e
n
(
c
o
m
m
o
n
)
,





}



d
e
f

s
e
l
e
c
t
_
p
r
o
v
i
d
e
r
(
)
:





a
v
a
i
l

=

o
r
t
.
g
e
t
_
a
v
a
i
l
a
b
l
e
_
p
r
o
v
i
d
e
r
s
(
)





i
f

'
C
U
D
A
E
x
e
c
u
t
i
o
n
P
r
o
v
i
d
e
r
'

i
n

a
v
a
i
l
:









r
e
t
u
r
n

[
'
C
U
D
A
E
x
e
c
u
t
i
o
n
P
r
o
v
i
d
e
r
'
,

'
C
P
U
E
x
e
c
u
t
i
o
n
P
r
o
v
i
d
e
r
'
]
,

'
c
u
d
a
'
,

'
C
U
D
A
E
x
e
c
u
t
i
o
n
P
r
o
v
i
d
e
r
'





r
e
t
u
r
n

[
'
C
P
U
E
x
e
c
u
t
i
o
n
P
r
o
v
i
d
e
r
'
]
,

'
c
p
u
'
,

'
C
P
U
E
x
e
c
u
t
i
o
n
P
r
o
v
i
d
e
r
'



d
e
f

g
r
e
e
d
y
_
d
e
c
o
d
e
_
o
n
n
x
(
e
n
c
_
s
e
s
s
,

d
e
c
_
i
n
i
t
_
s
e
s
s
,

d
e
c
_
p
a
s
t
_
s
e
s
s
,

























p
i
x
e
l
_
v
a
l
u
e
s
_
n
p
,

m
a
x
_
l
e
n
g
t
h
=
6
4
,

























n
o
_
r
e
p
e
a
t
_
n
g
r
a
m
_
s
i
z
e
=
3
,

r
e
p
e
t
i
t
i
o
n
_
p
e
n
a
l
t
y
=
1
.
3
)
:





"
"
"





G
r
e
e
d
y

d
e
c
o
d
e

v
ớ
i

K
V
-
c
a
c
h
e

+

r
e
p
e
t
i
t
i
o
n

p
e
n
a
l
t
y

+

n
o
-
r
e
p
e
a
t
-
n
g
r
a
m
.






T
h
a
m

s
ố

c
h
ố
n
g

l
ặ
p

(
q
u
a
n

t
r
ọ
n
g

v
ớ
i

I
N
T
8

q
u
a
n
t
i
z
e
d

d
e
c
o
d
e
r
)
:







n
o
_
r
e
p
e
a
t
_
n
g
r
a
m
_
s
i
z
e

:

c
h
ặ
n

s
i
n
h

l
ạ
i

n
-
g
r
a
m

đ
ã

x
u
ấ
t

h
i
ệ
n

(
m
ặ
c

đ
ị
n
h

3
)







r
e
p
e
t
i
t
i
o
n
_
p
e
n
a
l
t
y



:

g
i
ả
m

l
o
g
i
t

c
ủ
a

t
o
k
e
n

đ
ã

s
i
n
h

(
m
ặ
c

đ
ị
n
h

1
.
3
)









-

p
e
n
a
l
t
y

>

1

→

p
h
ạ
t

t
o
k
e
n

đ
ã

x
u
ấ
t

h
i
ệ
n

(
t
r
á
n
h

l
ặ
p
)









-

p
e
n
a
l
t
y

=

1

→

k
h
ô
n
g

p
h
ạ
t

(
h
à
n
h

v
i

g
r
e
e
d
y

t
h
u
ầ
n
)






Q
U
A
N

T
R
Ọ
N
G
:

h
à
m

k
h
ô
n
g

n
h
ậ
n

p
r
o
c
e
s
s
o
r

l
à
m

t
h
a
m

s
ố
.

















p
r
o
c
e
s
s
o
r

l
à

b
i
ế
n

g
l
o
b
a
l

t
r
o
n
g

n
o
t
e
b
o
o
k
.





"
"
"





B

=

p
i
x
e
l
_
v
a
l
u
e
s
_
n
p
.
s
h
a
p
e
[
0
]







1
.

V
i
s
i
o
n

e
n
c
o
d
e





e
n
c
_
h
i
d
d
e
n

=

e
n
c
_
s
e
s
s
.
r
u
n
(
N
o
n
e
,

{
'
p
i
x
e
l
_
v
a
l
u
e
s
'
:

p
i
x
e
l
_
v
a
l
u
e
s
_
n
p
}
)
[
0
]







2
.

D
e
c
o
d
e
r

i
n
i
t

(
B
O
S

t
o
k
e
n
,

k
h
ô
n
g

c
ó

p
a
s
t

K
V
)





g
e
n
e
r
a
t
e
d

=

n
p
.
f
u
l
l
(
(
B
,

1
)
,

b
o
s
_
t
o
k
e
n
_
i
d
,

d
t
y
p
e
=
n
p
.
i
n
t
6
4
)





f
i
n
i
s
h
e
d


=

n
p
.
z
e
r
o
s
(
B
,

d
t
y
p
e
=
b
o
o
l
)






i
n
i
t
_
o
u
t
s

=

d
e
c
_
i
n
i
t
_
s
e
s
s
.
r
u
n
(
N
o
n
e
,

{









'
i
n
p
u
t
_
i
d
s
'
:













g
e
n
e
r
a
t
e
d
,









'
a
t
t
e
n
t
i
o
n
_
m
a
s
k
'
:








n
p
.
o
n
e
s
_
l
i
k
e
(
g
e
n
e
r
a
t
e
d
,

d
t
y
p
e
=
n
p
.
i
n
t
6
4
)
,









'
e
n
c
o
d
e
r
_
h
i
d
d
e
n
_
s
t
a
t
e
s
'
:

e
n
c
_
h
i
d
d
e
n
,





}
)





l
o
g
i
t
s

=

i
n
i
t
_
o
u
t
s
[
0
]



(
B
,

1
,

v
o
c
a
b
)







M
a
p

o
u
t
p
u
t

p
r
e
s
_
*

-
>

p
a
s
t

d
i
c
t

c
h
o

b
ư
ớ
c

t
i
ế
p

t
h
e
o





p
a
s
t

=

{
}





f
o
r

i

i
n

r
a
n
g
e
(
n
u
m
_
l
a
y
e
r
s
)
:









p
a
s
t
[
f
'
p
a
s
t
_
s
e
l
f
_
k
e
y
_
{
i
}
'
]


=

i
n
i
t
_
o
u
t
s
[
1

+

i
*
4
]









p
a
s
t
[
f
'
p
a
s
t
_
s
e
l
f
_
v
a
l
_
{
i
}
'
]


=

i
n
i
t
_
o
u
t
s
[
2

+

i
*
4
]









p
a
s
t
[
f
'
p
a
s
t
_
c
r
o
s
s
_
k
e
y
_
{
i
}
'
]

=

i
n
i
t
_
o
u
t
s
[
3

+

i
*
4
]









p
a
s
t
[
f
'
p
a
s
t
_
c
r
o
s
s
_
v
a
l
_
{
i
}
'
]

=

i
n
i
t
_
o
u
t
s
[
4

+

i
*
4
]






n
e
x
t
_
t
o
k


=

l
o
g
i
t
s
[
:
,

-
1
,

:
]
.
a
r
g
m
a
x
(
-
1
)
.
a
s
t
y
p
e
(
n
p
.
i
n
t
6
4
)





n
e
x
t
_
t
o
k


=

n
p
.
w
h
e
r
e
(
f
i
n
i
s
h
e
d
,

p
a
d
_
t
o
k
e
n
_
i
d
,

n
e
x
t
_
t
o
k
)





g
e
n
e
r
a
t
e
d

=

n
p
.
c
o
n
c
a
t
e
n
a
t
e
(
[
g
e
n
e
r
a
t
e
d
,

n
e
x
t
_
t
o
k
[
:
,

N
o
n
e
]
]
,

a
x
i
s
=
1
)





f
i
n
i
s
h
e
d


=

f
i
n
i
s
h
e
d

|

(
n
e
x
t
_
t
o
k

=
=

e
o
s
_
t
o
k
e
n
_
i
d
)







H
e
l
p
e
r
:

á
p

r
e
p
e
t
i
t
i
o
n

p
e
n
a
l
t
y

l
ê
n

l
o
g
i
t

v
e
c
t
o
r





d
e
f

a
p
p
l
y
_
r
e
p
e
t
i
t
i
o
n
_
p
e
n
a
l
t
y
(
l
o
g
i
t
s
_
s
t
e
p
,

g
e
n
_
i
d
s
,

p
e
n
a
l
t
y
)
:









"
"
"
l
o
g
i
t
s
_
s
t
e
p
:

(
B
,

v
o
c
a
b
)
,

g
e
n
_
i
d
s
:

(
B
,

T
)
"
"
"









o
u
t

=

l
o
g
i
t
s
_
s
t
e
p
.
c
o
p
y
(
)









f
o
r

b

i
n

r
a
n
g
e
(
B
)
:













f
o
r

t
o
k

i
n

s
e
t
(
g
e
n
_
i
d
s
[
b
]
.
t
o
l
i
s
t
(
)
)
:

















i
f

o
u
t
[
b
,

t
o
k
]

<

0
:





















o
u
t
[
b
,

t
o
k
]

*
=

p
e
n
a
l
t
y

















e
l
s
e
:





















o
u
t
[
b
,

t
o
k
]

/
=

p
e
n
a
l
t
y









r
e
t
u
r
n

o
u
t







H
e
l
p
e
r
:

m
a
s
k

n
-
g
r
a
m

đ
ã

x
u
ấ
t

h
i
ệ
n





d
e
f

a
p
p
l
y
_
n
o
_
r
e
p
e
a
t
_
n
g
r
a
m
(
l
o
g
i
t
s
_
s
t
e
p
,

g
e
n
_
i
d
s
,

n
g
r
a
m
_
s
i
z
e
)
:









"
"
"
Đ
ặ
t

l
o
g
i
t

=

-
i
n
f

c
h
o

t
o
k
e
n

s
ẽ

t
ạ
o

n
-
g
r
a
m

đ
ã

t
ồ
n

t
ạ
i
.
"
"
"









o
u
t

=

l
o
g
i
t
s
_
s
t
e
p
.
c
o
p
y
(
)









i
f

n
g
r
a
m
_
s
i
z
e

<
=

0
:













r
e
t
u
r
n

o
u
t









f
o
r

b

i
n

r
a
n
g
e
(
B
)
:













s
e
q

=

g
e
n
_
i
d
s
[
b
]
.
t
o
l
i
s
t
(
)













i
f

l
e
n
(
s
e
q
)

<

n
g
r
a
m
_
s
i
z
e

-

1
:

















c
o
n
t
i
n
u
e














p
r
e
f
i
x

=

(
n
g
r
a
m
_
s
i
z
e
-
1
)

t
o
k
e
n

c
u
ố
i













p
r
e
f
i
x

=

t
u
p
l
e
(
s
e
q
[
-
(
n
g
r
a
m
_
s
i
z
e

-

1
)
:
]
)













b
a
n
n
e
d

=

s
e
t
(
)













f
o
r

s
t
a
r
t

i
n

r
a
n
g
e
(
l
e
n
(
s
e
q
)

-

n
g
r
a
m
_
s
i
z
e

+

1
)
:

















i
f

t
u
p
l
e
(
s
e
q
[
s
t
a
r
t
:
s
t
a
r
t

+

n
g
r
a
m
_
s
i
z
e

-

1
]
)

=
=

p
r
e
f
i
x
:





















b
a
n
n
e
d
.
a
d
d
(
s
e
q
[
s
t
a
r
t

+

n
g
r
a
m
_
s
i
z
e

-

1
]
)













f
o
r

t
o
k

i
n

b
a
n
n
e
d
:

















o
u
t
[
b
,

t
o
k
]

=

-
1
e
9









r
e
t
u
r
n

o
u
t







3
.

S
t
e
p

d
e
c
o
d
e

v
ớ
i

p
a
s
t

K
V





f
o
r

_

i
n

r
a
n
g
e
(
m
a
x
_
l
e
n
g
t
h

-

2
)
:









i
f

f
i
n
i
s
h
e
d
.
a
l
l
(
)
:













b
r
e
a
k









d
e
c
_
i
n

=

{













'
i
n
p
u
t
_
i
d
s
'
:






n
e
x
t
_
t
o
k
[
:
,

N
o
n
e
]
,













'
a
t
t
e
n
t
i
o
n
_
m
a
s
k
'
:

n
p
.
o
n
e
s
(
(
B
,

g
e
n
e
r
a
t
e
d
.
s
h
a
p
e
[
1
]
)
,

d
t
y
p
e
=
n
p
.
i
n
t
6
4
)
,









}









d
e
c
_
i
n
.
u
p
d
a
t
e
(
p
a
s
t
)









s
t
e
p
_
o
u
t
s

=

d
e
c
_
p
a
s
t
_
s
e
s
s
.
r
u
n
(
N
o
n
e
,

d
e
c
_
i
n
)









l
o
g
i
t
s




=

s
t
e
p
_
o
u
t
s
[
0
]









f
o
r

i

i
n

r
a
n
g
e
(
n
u
m
_
l
a
y
e
r
s
)
:













p
a
s
t
[
f
'
p
a
s
t
_
s
e
l
f
_
k
e
y
_
{
i
}
'
]


=

s
t
e
p
_
o
u
t
s
[
1

+

i
*
4
]













p
a
s
t
[
f
'
p
a
s
t
_
s
e
l
f
_
v
a
l
_
{
i
}
'
]


=

s
t
e
p
_
o
u
t
s
[
2

+

i
*
4
]













p
a
s
t
[
f
'
p
a
s
t
_
c
r
o
s
s
_
k
e
y
_
{
i
}
'
]

=

s
t
e
p
_
o
u
t
s
[
3

+

i
*
4
]













p
a
s
t
[
f
'
p
a
s
t
_
c
r
o
s
s
_
v
a
l
_
{
i
}
'
]

=

s
t
e
p
_
o
u
t
s
[
4

+

i
*
4
]











L
ấ
y

l
o
g
i
t

t
ạ
i

v
ị

t
r
í

c
u
ố
i









s
t
e
p
_
l
o
g
i
t
s

=

l
o
g
i
t
s
[
:
,

-
1
,

:
]



(
B
,

v
o
c
a
b
)











Á
p

r
e
p
e
t
i
t
i
o
n

p
e
n
a
l
t
y









i
f

r
e
p
e
t
i
t
i
o
n
_
p
e
n
a
l
t
y

!
=

1
.
0
:













s
t
e
p
_
l
o
g
i
t
s

=

a
p
p
l
y
_
r
e
p
e
t
i
t
i
o
n
_
p
e
n
a
l
t
y
(
s
t
e
p
_
l
o
g
i
t
s
,

g
e
n
e
r
a
t
e
d
,

r
e
p
e
t
i
t
i
o
n
_
p
e
n
a
l
t
y
)











Á
p

n
o
-
r
e
p
e
a
t
-
n
g
r
a
m









i
f

n
o
_
r
e
p
e
a
t
_
n
g
r
a
m
_
s
i
z
e

>

0
:













s
t
e
p
_
l
o
g
i
t
s

=

a
p
p
l
y
_
n
o
_
r
e
p
e
a
t
_
n
g
r
a
m
(
s
t
e
p
_
l
o
g
i
t
s
,

g
e
n
e
r
a
t
e
d
,

n
o
_
r
e
p
e
a
t
_
n
g
r
a
m
_
s
i
z
e
)










n
e
x
t
_
t
o
k


=

s
t
e
p
_
l
o
g
i
t
s
.
a
r
g
m
a
x
(
-
1
)
.
a
s
t
y
p
e
(
n
p
.
i
n
t
6
4
)









n
e
x
t
_
t
o
k


=

n
p
.
w
h
e
r
e
(
f
i
n
i
s
h
e
d
,

p
a
d
_
t
o
k
e
n
_
i
d
,

n
e
x
t
_
t
o
k
)









g
e
n
e
r
a
t
e
d

=

n
p
.
c
o
n
c
a
t
e
n
a
t
e
(
[
g
e
n
e
r
a
t
e
d
,

n
e
x
t
_
t
o
k
[
:
,

N
o
n
e
]
]
,

a
x
i
s
=
1
)









f
i
n
i
s
h
e
d


=

f
i
n
i
s
h
e
d

|

(
n
e
x
t
_
t
o
k

=
=

e
o
s
_
t
o
k
e
n
_
i
d
)






r
e
t
u
r
n

p
r
o
c
e
s
s
o
r
.
b
a
t
c
h
_
d
e
c
o
d
e
(
g
e
n
e
r
a
t
e
d
,

s
k
i
p
_
s
p
e
c
i
a
l
_
t
o
k
e
n
s
=
T
r
u
e
)



p
r
i
n
t
(
'
U
t
i
l
i
t
i
e
s

d
e
f
i
n
e
d
.
'
)


In [ ]:
p
r
o
v
i
d
e
r
_
l
i
s
t
,

b
e
n
c
h
_
d
e
v
i
c
e
,

p
r
o
v
i
d
e
r
_
n
a
m
e

=

s
e
l
e
c
t
_
p
r
o
v
i
d
e
r
(
)



B
ậ
t

g
r
a
p
h

o
p
t
i
m
i
z
a
t
i
o
n

t
h
a
y

c
h
o

o
p
t
i
m
i
z
e
_
m
o
d
e
l
=
T
r
u
e

đ
ã

b
ị

x
o
á

k
h
ỏ
i

q
u
a
n
t
i
z
e
_
d
y
n
a
m
i
c

s
e
s
s
_
o
p
t
s

=

o
r
t
.
S
e
s
s
i
o
n
O
p
t
i
o
n
s
(
)

s
e
s
s
_
o
p
t
s
.
g
r
a
p
h
_
o
p
t
i
m
i
z
a
t
i
o
n
_
l
e
v
e
l

=

o
r
t
.
G
r
a
p
h
O
p
t
i
m
i
z
a
t
i
o
n
L
e
v
e
l
.
O
R
T
_
E
N
A
B
L
E
_
A
L
L



V
i
s
i
o
n

e
n
c
o
d
e
r
:

F
P
3
2

s
e
s
s
i
o
n

(
k
h
ô
n
g

q
u
a
n
t
i
z
e
)

e
n
c
_
s
e
s
s






=

o
r
t
.
I
n
f
e
r
e
n
c
e
S
e
s
s
i
o
n
(
E
N
C
O
D
E
R
_
I
N
T
8
_
P
A
T
H
,


s
e
s
s
_
o
p
t
s
,

p
r
o
v
i
d
e
r
s
=
p
r
o
v
i
d
e
r
_
l
i
s
t
)


T
e
x
t

d
e
c
o
d
e
r
s
:

I
N
T
8

s
e
s
s
i
o
n
s

d
e
c
_
i
n
i
t
_
s
e
s
s

=

o
r
t
.
I
n
f
e
r
e
n
c
e
S
e
s
s
i
o
n
(
D
E
C
_
I
N
I
T
_
I
N
T
8
_
P
A
T
H
,

s
e
s
s
_
o
p
t
s
,

p
r
o
v
i
d
e
r
s
=
p
r
o
v
i
d
e
r
_
l
i
s
t
)

d
e
c
_
p
a
s
t
_
s
e
s
s

=

o
r
t
.
I
n
f
e
r
e
n
c
e
S
e
s
s
i
o
n
(
D
E
C
_
P
A
S
T
_
I
N
T
8
_
P
A
T
H
,

s
e
s
s
_
o
p
t
s
,

p
r
o
v
i
d
e
r
s
=
p
r
o
v
i
d
e
r
_
l
i
s
t
)


p
r
i
n
t
(
f
'
P
r
o
v
i
d
e
r
:

{
p
r
o
v
i
d
e
r
_
n
a
m
e
}


|


D
e
v
i
c
e
:

{
b
e
n
c
h
_
d
e
v
i
c
e
}
'
)

p
r
i
n
t
(
f
'
V
i
s
i
o
n

e
n
c
o
d
e
r

:

F
P
3
2


(
{
o
s
.
p
a
t
h
.
b
a
s
e
n
a
m
e
(
E
N
C
O
D
E
R
_
I
N
T
8
_
P
A
T
H
)
}
)
'
)

p
r
i
n
t
(
f
'
D
e
c
o
d
e
r

i
n
i
t



:

I
N
T
8


(
{
o
s
.
p
a
t
h
.
b
a
s
e
n
a
m
e
(
D
E
C
_
I
N
I
T
_
I
N
T
8
_
P
A
T
H
)
}
)
'
)

p
r
i
n
t
(
f
'
D
e
c
o
d
e
r

p
a
s
t



:

I
N
T
8


(
{
o
s
.
p
a
t
h
.
b
a
s
e
n
a
m
e
(
D
E
C
_
P
A
S
T
_
I
N
T
8
_
P
A
T
H
)
}
)
'
)


In [ ]:
t
e
s
t
_
d
a
t
a
s
e
t

=

U
I
T
V
i
I
C
D
a
t
a
s
e
t
(
T
E
S
T
_
P
A
T
H
,

I
M
A
G
E
S
_
D
I
R
,

p
r
o
c
e
s
s
o
r
,

M
A
X
_
L
E
N
G
T
H
)

t
e
s
t
_
d
a
t
a
s
e
t
.
i
m
a
g
e
_
g
r
o
u
p
s

=

t
e
s
t
_
d
a
t
a
s
e
t
.
i
m
a
g
e
_
g
r
o
u
p
s
.
h
e
a
d
(
M
A
X
_
T
E
S
T
_
I
M
A
G
E
S
)
.
r
e
s
e
t
_
i
n
d
e
x
(
d
r
o
p
=
T
r
u
e
)

t
e
s
t
_
l
o
a
d
e
r


=

D
a
t
a
L
o
a
d
e
r
(
t
e
s
t
_
d
a
t
a
s
e
t
,

b
a
t
c
h
_
s
i
z
e
=
B
A
T
C
H
_
S
I
Z
E
,



























s
h
u
f
f
l
e
=
F
a
l
s
e
,

n
u
m
_
w
o
r
k
e
r
s
=
N
U
M
_
W
O
R
K
E
R
S
,

c
o
l
l
a
t
e
_
f
n
=
c
o
l
l
a
t
e
_
f
n
)


r
e
f
s
_
d
i
c
t

=

b
u
i
l
d
_
r
e
f
e
r
e
n
c
e
s
(
t
e
s
t
_
l
o
a
d
e
r
)

p
r
i
n
t
(
f
'
T
e
s
t

i
m
a
g
e
s
:

{
l
e
n
(
t
e
s
t
_
d
a
t
a
s
e
t
)
}


|


b
a
t
c
h
e
s
:

{
l
e
n
(
t
e
s
t
_
l
o
a
d
e
r
)
}
'
)



W
a
r
m
u
p

—

t
r
á
n
h

t
í
n
h

t
h
ờ
i

g
i
a
n

J
I
T
/
c
a
c
h
e

c
o
l
d
-
s
t
a
r
t

v
à
o

l
a
t
e
n
c
y

s
a
m
p
l
e
_
p
x

=

p
r
o
c
e
s
s
o
r
(





i
m
a
g
e
s
=
I
m
a
g
e
.
o
p
e
n
(
S
A
M
P
L
E
_
I
M
A
G
E
)
.
c
o
n
v
e
r
t
(
'
R
G
B
'
)
,

r
e
t
u
r
n
_
t
e
n
s
o
r
s
=
'
p
t
'

)
[
'
p
i
x
e
l
_
v
a
l
u
e
s
'
]
.
n
u
m
p
y
(
)
.
a
s
t
y
p
e
(
n
p
.
f
l
o
a
t
3
2
)

_

=

g
r
e
e
d
y
_
d
e
c
o
d
e
_
o
n
n
x
(
e
n
c
_
s
e
s
s
,

d
e
c
_
i
n
i
t
_
s
e
s
s
,

d
e
c
_
p
a
s
t
_
s
e
s
s
,

s
a
m
p
l
e
_
p
x
)

p
r
i
n
t
(
'
W
a
r
m
u
p

d
o
n
e
.
'
)


In [ ]:
p
r
e
d
s
_
d
i
c
t

=

{
}

l
a
t
e
n
c
i
e
s


=

[
]


f
o
r

b
a
t
c
h
_
i
d
x
,

b
a
t
c
h

i
n

e
n
u
m
e
r
a
t
e
(
t
q
d
m
(
t
e
s
t
_
l
o
a
d
e
r
,

d
e
s
c
=
'
B
e
n
c
h
m
a
r
k
i
n
g

O
N
N
X

I
N
T
8
'
)
)
:





p
x

=

b
a
t
c
h
[
'
p
i
x
e
l
_
v
a
l
u
e
s
'
]
.
n
u
m
p
y
(
)
.
a
s
t
y
p
e
(
n
p
.
f
l
o
a
t
3
2
)






t
0




=

t
i
m
e
.
p
e
r
f
_
c
o
u
n
t
e
r
(
)





t
e
x
t
s

=

g
r
e
e
d
y
_
d
e
c
o
d
e
_
o
n
n
x
(
e
n
c
_
s
e
s
s
,

d
e
c
_
i
n
i
t
_
s
e
s
s
,

d
e
c
_
p
a
s
t
_
s
e
s
s
,

p
x
,

M
A
X
_
L
E
N
G
T
H
)





t
1




=

t
i
m
e
.
p
e
r
f
_
c
o
u
n
t
e
r
(
)






i
f

b
a
t
c
h
_
i
d
x

>
=

W
A
R
M
U
P
_
S
T
E
P
S
:









l
a
t
e
n
c
i
e
s
.
a
p
p
e
n
d
(
t
1

-

t
0
)






f
o
r

i
m
g
_
f
,

t
e
x
t

i
n

z
i
p
(
b
a
t
c
h
[
'
i
m
a
g
e
_
f
i
l
e
'
]
,

t
e
x
t
s
)
:









p
r
e
d
s
_
d
i
c
t
[
i
m
g
_
f
]

=

t
e
x
t
.
s
t
r
i
p
(
)


m
e
t
r
i
c
s




=

c
a
l
c
u
l
a
t
e
_
m
e
t
r
i
c
s
(
p
r
e
d
s
_
d
i
c
t
,

r
e
f
s
_
d
i
c
t
)

a
v
g
_
l
a
t




=

f
l
o
a
t
(
n
p
.
m
e
a
n
(
l
a
t
e
n
c
i
e
s
)
)









i
f

l
a
t
e
n
c
i
e
s

e
l
s
e

0
.
0

p
9
5
_
l
a
t




=

f
l
o
a
t
(
n
p
.
p
e
r
c
e
n
t
i
l
e
(
l
a
t
e
n
c
i
e
s
,

9
5
)
)

i
f

l
a
t
e
n
c
i
e
s

e
l
s
e

0
.
0

t
h
r
o
u
g
h
p
u
t

=

f
l
o
a
t
(
B
A
T
C
H
_
S
I
Z
E

/

a
v
g
_
l
a
t
)







i
f

a
v
g
_
l
a
t

>

0

e
l
s
e

0
.
0


b
e
n
c
h
m
a
r
k

=

{





'
n
a
m
e
'
:



























'
o
n
n
x
_
i
n
t
8
'
,





'
b
a
c
k
e
n
d
'
:
























'
o
n
n
x
'
,





'
p
r
e
c
i
s
i
o
n
'
:






















'
i
n
t
8
'
,





'
d
e
v
i
c
e
'
:

























b
e
n
c
h
_
d
e
v
i
c
e
,





'
p
r
o
v
i
d
e
r
'
:























p
r
o
v
i
d
e
r
_
n
a
m
e
,





'
g
e
n
e
r
a
t
i
o
n
_
s
t
r
a
t
e
g
y
'
:












'
g
r
e
e
d
y
+
k
v
c
a
c
h
e
'
,





'
c
a
c
h
e
_
e
n
a
b
l
e
d
'
:


















T
r
u
e
,





'
q
u
a
n
t
i
z
a
t
i
o
n
_
s
c
o
p
e
'
:













'
d
e
c
o
d
e
r
_
o
n
l
y
'
,





'
q
u
a
n
t
i
z
a
t
i
o
n
_
m
e
t
h
o
d
'
:












'
d
y
n
a
m
i
c
_
w
e
i
g
h
t
_
o
n
l
y
'
,





'
r
e
d
u
c
e
_
r
a
n
g
e
'
:



















T
r
u
e
,





'
b
a
t
c
h
_
s
i
z
e
'
:





















B
A
T
C
H
_
S
I
Z
E
,





'
m
a
x
_
t
e
s
t
_
i
m
a
g
e
s
'
:
















M
A
X
_
T
E
S
T
_
I
M
A
G
E
S
,





'
a
v
g
_
l
a
t
e
n
c
y
_
s
e
c
o
n
d
s
_
p
e
r
_
b
a
t
c
h
'
:


a
v
g
_
l
a
t
,





'
p
9
5
_
l
a
t
e
n
c
y
_
s
e
c
o
n
d
s
_
p
e
r
_
b
a
t
c
h
'
:


p
9
5
_
l
a
t
,





'
t
h
r
o
u
g
h
p
u
t
_
i
m
a
g
e
s
_
p
e
r
_
s
e
c
o
n
d
'
:



t
h
r
o
u
g
h
p
u
t
,





'
b
l
e
u
4
'
:


























m
e
t
r
i
c
s
[
'
b
l
e
u
4
'
]
,





'
r
o
u
g
e
L
'
:

























m
e
t
r
i
c
s
[
'
r
o
u
g
e
L
'
]
,





'
m
e
t
e
o
r
'
:

























m
e
t
r
i
c
s
[
'
m
e
t
e
o
r
'
]
,





'
n
u
m
_
i
m
a
g
e
s
'
:





















m
e
t
r
i
c
s
[
'
n
u
m
_
i
m
a
g
e
s
'
]
,





'
m
o
d
e
l
_
p
a
t
h
s
'
:

{









'
v
i
s
i
o
n
_
e
n
c
o
d
e
r
'
:




E
N
C
O
D
E
R
_
I
N
T
8
_
P
A
T
H
,









'
d
e
c
o
d
e
r
_
i
n
i
t
'
:






D
E
C
_
I
N
I
T
_
I
N
T
8
_
P
A
T
H
,









'
d
e
c
o
d
e
r
_
w
i
t
h
_
p
a
s
t
'
:

D
E
C
_
P
A
S
T
_
I
N
T
8
_
P
A
T
H
,





}
,

}


p
r
i
n
t
(
j
s
o
n
.
d
u
m
p
s
(
b
e
n
c
h
m
a
r
k
,

i
n
d
e
n
t
=
2
,

e
n
s
u
r
e
_
a
s
c
i
i
=
F
a
l
s
e
)
)


w
i
t
h

o
p
e
n
(
o
s
.
p
a
t
h
.
j
o
i
n
(
R
E
S
U
L
T
S
_
D
I
R
,

'
o
n
n
x
_
i
n
t
8
.
j
s
o
n
'
)
,

'
w
'
,

e
n
c
o
d
i
n
g
=
'
u
t
f
-
8
'
)

a
s

f
:





j
s
o
n
.
d
u
m
p
(
b
e
n
c
h
m
a
r
k
,

f
,

e
n
s
u
r
e
_
a
s
c
i
i
=
F
a
l
s
e
,

i
n
d
e
n
t
=
2
)

p
r
i
n
t
(
"
\
n
S
a
v
e
d
:

r
e
s
u
l
t
s
/
o
n
n
x
_
i
n
t
8
.
j
s
o
n
"
)
